In [ ]:
!git clone https://github.com/zhao-zilong/ssc-cot.git


In [ ]:
!pip install -q peft transformers datasets accelerate bitsandbytes trl


In [ ]:
import os
files = os.listdir("/content/ssc-cot/Dataset")
print(len(files))


In [ ]:
import os
import json

dataset_path = "/content/ssc-cot/Dataset"

all_data = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".json"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    if isinstance(data, list):
                        all_data.extend(data)
                    elif isinstance(data, dict):
                        all_data.append(data)
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

print(f"Total samples loaded: {len(all_data)}")


In [ ]:
print(all_data[0])
print(all_data[0].keys())


In [ ]:
from datasets import Dataset

STEP_SEPARATOR = " ки"
POSITIVE_TOKEN = "+"
NEGATIVE_TOKEN = "-"

def convert_prm(sample):
    question = sample.get("question", "")
    intermediate = sample.get("intermediate-results", [])
    steps = [s.get("result", "") for s in intermediate]
    final_result = sample.get("result", None)
    last_step_result = intermediate[-1].get("result", "") if intermediate else ""
    solution_correct = (str(final_result).strip() == str(last_step_result).strip()) if final_result else True
    if solution_correct:
        labels = [POSITIVE_TOKEN] * len(steps)
    else:
        labels = [POSITIVE_TOKEN] * (len(steps) - 1) + [NEGATIVE_TOKEN]
    return {
        "question": question,
        "steps": steps,
        "labels": labels
    }

def format_prm_text(sample):
    question = sample["question"]
    steps = sample["steps"]
    labels = sample["labels"]
    text = question + "\n"
    for step, label in zip(steps, labels):
        text += step + STEP_SEPARATOR + label + "\n"
    return {"text": text}

hf_data = [convert_prm(x) for x in all_data if isinstance(x, dict) and x.get("intermediate-results")]
dataset = Dataset.from_list(hf_data)
dataset = dataset.map(format_prm_text)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print(dataset["train"][0]["text"][:500])


In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Free any leftover memory before loading
gc.collect()
torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit NF4 quantization — keeps Qwen2.5-7B well within T4 16 GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 does not support bfloat16
    bnb_4bit_use_double_quant=True,          # extra ~0.4 bpp saving
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded successfully")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Required when using 4-bit quantization
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,                       # Reduced from 16 → saves VRAM during backward pass
    lora_alpha=16,             # Scaled proportionally with r
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Reduced from 512 → 256 to halve activation memory on T4
MAX_LENGTH = 256

def tokenize(sample):
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = dataset["train"].map(
    tokenize, batched=True,
    remove_columns=dataset["train"].column_names,
    batch_size=64   # smaller map batch → less RAM spike
)
tokenized_eval = dataset["test"].map(
    tokenize, batched=True,
    remove_columns=dataset["test"].column_names,
    batch_size=64
)

print(tokenized_train)


In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import torch, gc

gc.collect()
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./prm_lora_output",

    # ── batch / accumulation ──────────────────────────────────────
    per_device_train_batch_size=1,      # T4 can only fit 1 at seq_len=256
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,     # effective batch = 16; was 8 with larger batches

    num_train_epochs=3,

    # ── learning rate ─────────────────────────────────────────────
    learning_rate=2e-4,
    warmup_steps=20,

    # ── precision — T4 does NOT support bfloat16 ─────────────────
    fp16=True,
    bf16=False,

    # ── logging / eval / save ─────────────────────────────────────
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,

    # ── memory optimizations ──────────────────────────────────────
    gradient_checkpointing=True,        # trade compute for VRAM
    dataloader_pin_memory=False,        # saves CPU↔GPU transfer memory
    dataloader_num_workers=0,           # avoids forked-process RAM overhead in Colab
    optim="paged_adamw_8bit",           # 8-bit paged optimizer — huge VRAM saver
    group_by_length=True,               # pack similar-length sequences → less padding waste

    report_to="none",
    remove_unused_columns=False,

    # ── prevent OOM at eval ───────────────────────────────────────
    prediction_loss_only=True,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,   # aligns tensors for Tensor Core efficiency
)

model.config.use_cache = False   # must be False with gradient_checkpointing

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

trainer.train()


In [ ]:
model.save_pretrained("./prm_lora_final")
tokenizer.save_pretrained("./prm_lora_final")
print("Model saved to ./prm_lora_final")


In [ ]:
model_path = "/content/prm_lora_final"


In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Free training memory before inference
gc.collect()
torch.cuda.empty_cache()

base_model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model in 4-bit, then merge LoRA adapter on top
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"Inference model ready | GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
prompt = """You are a math solution grader.

DO NOT solve the problem again.
ONLY evaluate the given steps.

Scoring rule (VERY IMPORTANT):
- Score each step from 0–10 based on BOTH correctness AND importance.
- Importance = how critical the step is to reaching the final answer.

Guidelines:
- Minor algebra / identity expansion → low importance (1–3)
- Intermediate simplifications → medium importance (4–6)
- Key transformation / insight → high importance (7–9)
- Final conclusion / decisive step → highest importance (10)

If a step is correct but trivial → give LOW score.
If a step is critical for solving → give HIGH score.

Question:
sinA - cosB = 3cosA - 3sinB and sin(A+B) ≠ 1, find sin(A-B)

Solution to grade:
1. sinA - cosB = 3cosA - 3sinB
2. ⇒ sinA - 3cosA = 3sinB - cosB
3. ⇒ sinA - 3cosA = √10 sin(A - φ), tanφ = 3
4. ⇒ 3sinB - cosB = √10 sin(B - φ)
5. ⇒ sin(A - φ) = sin(B - φ)
6. ⇒ A = B or A + B = π
7. ⇒ sin(A - B) = 0

Task:
For EACH step output:
- correctness: correct / incorrect
- importance: low / medium / high
- explanation: one short line
- score: (0–10)

Then output FINAL SCORE based on overall reasoning quality.

Output format (STRICT):

Step 1:
correctness:
importance:
explanation:
score:

Step 2:
...

Score formula:
score = importance_weight × correctness

Where:
low = 2
medium = 5
high = 8–10
"""


In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")


In [ ]:
with torch.no_grad():   # essential — disables grad tracking during inference
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,         # reduced from 1024; enough for step-by-step grading
        temperature=0.1,
        top_p=1.0,
        do_sample=False,
        repetition_penalty=1.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )


In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
